Loading LINZ Data as job to shared drive

In [0]:
%pip install google-api-python-client

import logging
import requests
import utility_api_linz
import json
import io
from observability_logging import DeltaTableHandler
from datetime import datetime
from google.oauth2 import service_account
from googleapiclient.http import MediaFileUpload
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseUpload

# load logger
logger = logging.getLogger("TableLogger")
# Clear existing handlers to prevent duplication
logger.handlers.clear()
handler = DeltaTableHandler("default.application_logs")
logger.addHandler(handler)
logger.setLevel(logging.INFO)

#load auth json
json_key_str = dbutils.secrets.get(scope="apikle_shared_drive", key="info")
info = json.loads(json_key_str)

#Authenticate and build service
SCOPES = ['https://www.googleapis.com/auth/drive']
PARENT_FOLDER_ID = '17NnUwQYh6lvHrk41Cbvju-PgtqwTejKu'
creds = service_account.Credentials.from_service_account_info(info, scopes=SCOPES)
service = build('drive', 'v3', credentials=creds)

try:    
    task_run_id = dbutils.widgets.get("task_run_id")
except Exception:
    task_run_id = None

print(f"Captured Run ID: {task_run_id}")

def job_ingest():
    try:
        logger.info("job_ingest start")
        logger.info(task_run_id)
        response = utility_api_linz.utility_api_linz_get('2026-04-20','2026-04-26', 'layer-113968-changeset')
        for row in response['data']['features']:
            datetime_now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            file_name = f'layer-113968-changeset_{datetime_now}.json'
            json_content = json.dumps(row, indent=4)
            file_metadata = {'name': file_name, 'parents': ['17NnUwQYh6lvHrk41Cbvju-PgtqwTejKu']}
            media = MediaIoBaseUpload(io.BytesIO(json_content.encode()), mimetype='application/json')
            file = service.files().create(body=file_metadata, media_body=media, fields='id', supportsAllDrives=True).execute()
        logger.info("job_ingest end")
    except Exception as e:
        print(e)

job_ingest()